# Handwritten Digit Recognition using Artificial Neural Networks (ANN)

**Author:** Sushant Tiwari
**Dataset:** [MNIST Handwritten Digits Dataset (Kaggle)](https://www.kaggle.com/datasets/oddrationale/mnist-in-csv)

**Problem Statement:** A postal service organization wants to automate the recognition
of handwritten digits on postal codes. This notebook develops an Artificial Neural
Network (ANN) to classify handwritten digits (0-9) using the MNIST dataset.

> Before running this notebook, download `mnist_train.csv` and `mnist_test.csv` from
> the Kaggle link above and place them inside a `data/` folder next to this notebook.

---
&copy; 2026 Sushant Tiwari. All Rights Reserved.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

DATA_DIR = "data"
OUTPUT_DIR = "outputs"
MODEL_DIR = "saved_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_STATE = 42


## Task 1: Data Understanding

1. Load the dataset using Pandas
2. Display the first five records
3. Identify input features and target variable
4. Display dataset dimensions and summary information
5. Display one sample handwritten digit using Matplotlib


In [ ]:
train_path = os.path.join(DATA_DIR, "mnist_train.csv")
test_path = os.path.join(DATA_DIR, "mnist_test.csv")

frames = []
if os.path.exists(train_path):
    frames.append(pd.read_csv(train_path))
if os.path.exists(test_path):
    frames.append(pd.read_csv(test_path))

assert frames, "Place mnist_train.csv / mnist_test.csv inside the data/ folder first."

df = pd.concat(frames, ignore_index=True)
df.head()


In [ ]:
print("Dataset dimensions (rows, columns):", df.shape)
df.info()


**Input features:** `pixel0` ... `pixel783` — 784 grayscale pixel intensity columns
(values 0-255), representing a flattened 28x28 handwritten digit image.

**Target variable:** `label` — the digit (0-9) the image represents.


In [ ]:
sample_row = df.iloc[0]
sample_label = sample_row["label"]
sample_image = sample_row.drop("label").values.reshape(28, 28).astype("uint8")

plt.figure(figsize=(3, 3))
plt.imshow(sample_image, cmap="gray")
plt.title(f"Sample Digit - Label: {sample_label}")
plt.axis("off")
plt.savefig(os.path.join(OUTPUT_DIR, "sample_digit.png"))
plt.show()


## Task 2: Data Preprocessing

- Check for missing values
- Separate features and target variable
- Normalize pixel values to the range 0-1
- Split the dataset into 80% training and 20% testing
- Convert the target labels into categorical format using One-Hot Encoding


In [ ]:
print("Total missing values:", df.isnull().sum().sum())

X = df.drop("label", axis=1).values
y = df["label"].values

# Normalize pixel values to 0-1
X = X.astype("float32") / 255.0

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Training samples:", X_train.shape[0], "| Testing samples:", X_test.shape[0])

# One-hot encode labels
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)
y_train_cat.shape


## Task 3: Model Development

Architecture:
- Input Layer (784 features)
- Hidden Layer 1: 128 neurons (ReLU)
- Hidden Layer 2: 64 neurons (ReLU)
- Output Layer: 10 neurons (Softmax)

Compiled with the Adam optimizer, categorical crossentropy loss, and accuracy metric.
Trained for 10 epochs.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(784,), name="Input_Layer"),
    layers.Dense(128, activation="relu", name="Hidden_Layer_1"),
    layers.Dense(64, activation="relu", name="Hidden_Layer_2"),
    layers.Dense(10, activation="softmax", name="Output_Layer"),
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


In [ ]:
history = model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=2,
)


In [ ]:
# Predict on the test dataset
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_pred[:10]


## Task 4: Model Evaluation

- Test Accuracy
- Confusion Matrix
- Classification Report
- Accuracy vs Epoch graph
- Loss vs Epoch graph


In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"))
plt.show()


In [ ]:
report = classification_report(y_test, y_pred, digits=4)
print(report)

with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w") as f:
    f.write(f"Test Accuracy: {test_accuracy*100:.2f}%\nTest Loss: {test_loss:.4f}\n\n")
    f.write(report)


In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy", marker="o")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy", marker="o")
plt.title("Accuracy vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(OUTPUT_DIR, "accuracy_vs_epoch.png"))
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(history.history["loss"], label="Training Loss", marker="o")
plt.plot(history.history["val_loss"], label="Validation Loss", marker="o")
plt.title("Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(OUTPUT_DIR, "loss_vs_epoch.png"))
plt.show()


In [ ]:
model_path = os.path.join(MODEL_DIR, "mnist_ann_model.keras")
model.save(model_path)
print("Model saved to:", model_path)


### Observations

1. The ANN converges quickly on MNIST, with training accuracy climbing faster
   than validation accuracy across the 10 epochs, reflecting how tractable this
   dataset is for a simple fully-connected network.
2. Misclassifications concentrate on visually similar digit pairs (e.g. 4 vs 9,
   3 vs 5, 7 vs 1), since a plain ANN treats pixels independently and ignores the
   2D spatial structure a CNN would exploit.
3. Precision and recall are fairly uniform across the ten classes in the
   classification report, showing no strong class-level bias.
4. If validation loss plateaus or rises while training loss keeps falling in the
   later epochs, that is an early sign of overfitting worth watching if training
   is extended beyond 10 epochs.

*(Re-verify these once you run the notebook on the full dataset — exact numbers
will depend on your run.)*


## Task 5: Conclusion

This project demonstrated that a simple Artificial Neural Network with two hidden
layers can classify handwritten digits from the MNIST dataset with high accuracy,
making it a viable approach for automating postal code digit recognition. The
hidden layers are what give the network its representational power — each layer
learns progressively more abstract combinations of pixel intensities, letting the
model separate classes that are not linearly separable in raw pixel space; without
them, the network would reduce to a single linear classifier. Compared to
traditional Machine Learning models (e.g. SVM, Random Forest), Deep Learning
architectures like this ANN automatically learn useful feature representations
directly from raw pixels, removing the need for manual feature engineering. A
key limitation of this plain ANN, however, is that it flattens the image and
ignores spatial relationships between neighboring pixels, which caps its accuracy
and robustness (e.g. to shifted or rotated digits) compared to a Convolutional
Neural Network purpose-built for image data.

---
&copy; 2026 Sushant Tiwari. All Rights Reserved.
